# 7. Modelos de Sub-word Tokenization y Modelos del Lenguaje Neuronales

## Objetivos

- Entrenar modelos para sub-word tokenization
  - Aplicar BPE a corpus

In [5]:
import re
import nltk
from rich import print as rprint

### Función de preprocesamiento

In [7]:
def preprocess(words: list[str], regex: str=r"[^\w+]", lang: str="en", remove_stops: bool = False, remove_accents: bool = False) -> list[str]:
    """Preprocess step for list of words in corpus
    """
    stop_lang = "english" if lang=="en" else "spanish"
    result = []
    for word in words:
        word = re.sub(regex, "", word).lower()
        if remove_stops and word in stopwords.words(stop_lang) or (len(word) < 2):
            continue
        
        if remove_accents:
            word = strip_accents(word)
        result.append(word)
    return result

## Vamos a tokenizar 🌈
![](https://i.pinimg.com/736x/58/6b/88/586b8825f010ce0e3f9c831f568aafa8.jpg)

In [8]:
BASE_PATH = "."
CORPORA_PATH = f"{BASE_PATH}/data/tokenization"
MODELS_PATH = f"{BASE_PATH}/models/sub-word"

#### Corpus en español: CESS

In [9]:
nltk.download("cess_esp")

[nltk_data] Downloading package cess_esp to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package cess_esp is already up-to-date!


True

In [10]:
from nltk.corpus import cess_esp

cess_words = cess_esp.words()

In [11]:
" ".join(cess_words[:30])

'El grupo estatal Electricité_de_France -Fpa- EDF -Fpt- anunció hoy , jueves , la compra del 51_por_ciento de la empresa mexicana Electricidad_Águila_de_Altamira -Fpa- EAA -Fpt- , creada por el japonés Mitsubishi_Corporation'

In [12]:
cess_plain_text = " ".join(preprocess(cess_words))

In [13]:
rprint(f"'{cess_plain_text[300:600]}'")

'que el proyecto para la construcción de altamira_2 al norte de tampico prevé la utilización de gas natural como 
combustible principal en una central de ciclo combinado que debe empezar funcionar en mayo_del_2002 la electricidad
producida pasará la red eléctrica pública de méxico en_virtud_de un acue'

In [ ]:
cess_preprocessed_words = cess_plain_text.split()

In [ ]:
with open(f"{CORPORA_PATH}/cess_plain.txt", "w") as f:
    f.write(cess_plain_text)

#### Corpus Inglés: Gutenberg

In [ ]:
nltk.download("punkt_tab")

In [ ]:
from nltk.corpus import gutenberg

gutenberg_words = gutenberg.words()[:200000]

In [ ]:
rprint(" ".join(gutenberg_words[:30]))

In [ ]:
gutenberg_plain_text = " ".join(preprocess(gutenberg_words))

rprint(gutenberg_plain_text[:100])

In [ ]:
gutenberg_preprocessed_words = gutenberg_plain_text.split()

In [ ]:
with open(f"{CORPORA_PATH}/gutenberg_plain.txt", "w") as f:
    f.write(gutenberg_plain_text)

#### Tokenizando el español con Hugging face

In [ ]:
from transformers import AutoTokenizer

spanish_tokenizer = AutoTokenizer.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-uncased"
)

In [ ]:
rprint(spanish_tokenizer.tokenize(cess_plain_text[1000:1400]))

In [ ]:
cess_types = Counter(cess_words)

In [ ]:
rprint(cess_types.most_common(10))

In [ ]:
cess_tokenized = spanish_tokenizer.tokenize(cess_plain_text)
rprint(cess_tokenized[:10])
cess_tokenized_types = Counter(cess_tokenized)

In [ ]:
rprint(cess_tokenized_types.most_common(30))

In [ ]:
cess_lemmatized_types = Counter(lemmatize(cess_words, lang="es"))

In [ ]:
rprint(cess_lemmatized_types.most_common(30))

In [ ]:
rprint("CESS")
rprint(f"Tipos ([bright_magenta]word-base[/]): {len(cess_types)}")
rprint(f"Tipos ([bright_yellow]lemmatized[/]): {len(cess_lemmatized_types)}")
rprint(f"Tipos ([bright_green]sub-word[/]): {len(cess_tokenized_types)}")

#### Tokenizando para el inglés

In [ ]:
gutenberg_types = Counter(gutenberg_words)

In [ ]:
gutenberg_tokenized = wp_tokenizer.tokenize(gutenberg_plain_text)
gutenberg_tokenized_types = Counter(gutenberg_tokenized)

In [ ]:
rprint(gutenberg_tokenized_types.most_common(10))

In [ ]:
gutenberg_lemmatized_types = Counter(lemmatize(gutenberg_preprocessed_words))

In [ ]:
rprint(gutenberg_lemmatized_types.most_common(20))

In [ ]:
rprint("Gutenberg")
rprint(f"Tipos ([bright_magenta]word-base[/]): {len(gutenberg_types)}")
rprint(f"Tipos ([bright_yellow]lemmatized[/]): {len(gutenberg_lemmatized_types)}")
rprint(f"Tipos ([bright_green]sub-word[/]): {len(gutenberg_tokenized_types)}")

#### OOV: out of vocabulary

Palabras que se vieron en el entrenamiento pero no estan en el test

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    gutenberg_words, test_size=0.3, random_state=42
)
rprint(len(train_data), len(test_data))

In [ ]:
s_1 = {"a", "b", "c", "d", "e"}
s_2 = {"a", "x", "y", "d"}
rprint(s_1 - s_2)
rprint(s_2 - s_1)

In [ ]:
oov_test = set(test_data) - set(train_data)

In [ ]:
for word in list(oov_test)[:3]:
    rprint(f"{word} in train: {word in set(train_data)}")

In [ ]:
train_tokenized, test_tokenized = train_test_split(
    gutenberg_tokenized, test_size=0.3, random_state=42
)
rprint(len(train_tokenized), len(test_tokenized))

In [ ]:
oov_tokenized_test = set(test_tokenized) - set(train_tokenized)

In [ ]:
rprint("OOV ([yellow]word-base):", len(oov_test))
rprint("OOV ([green]sub-word):", len(oov_tokenized_test))

## Entrenando nuestro modelo con BPE
![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmedia1.tenor.com%2Fimages%2Fd565618bb1217a7c435579d9172270d0%2Ftenor.gif%3Fitemid%3D3379322&f=1&nofb=1&ipt=9719714edb643995ce9d978c8bab77f5310204960093070e37e183d5372096d9&ipo=images)

In [ ]:
!pip install subword-nmt

In [17]:
!ls {CORPORA_PATH}

cess_plain.txt		 eng-bible.txt	      spa-bible-tokenized.txt
eng-bible-tokenized.txt  gutenberg_plain.txt  spa-bible.txt


In [18]:
!head -c 1000 {CORPORA_PATH}/gutenberg_plain.txt

emma by jane austen 1816 volume chapter emma woodhouse handsome clever and rich with comfortable home and happy disposition seemed to unite some of the best blessings of existence and had lived nearly twenty one years in the world with very little to distress or vex her she was the youngest of the two daughters of most affectionate indulgent father and had in consequence of her sister marriage been mistress of his house from very early period her mother had died too long ago for her to have more than an indistinct remembrance of her caresses and her place had been supplied by an excellent woman as governess who had fallen little short of mother in affection sixteen years had miss taylor been in mr woodhouse family less as governess than friend very fond of both daughters but particularly of emma between _them_ it was more the intimacy of sisters even before miss taylor had ceased to hold the nominal office of governess the mildness of her temper had hardly allowed her to impose any res

In [ ]:
!subword-nmt learn-bpe -s 300 < \
 {CORPORA_PATH}/gutenberg_plain.txt > \
  {MODELS_PATH}/gutenberg.model

In [ ]:
!echo "I need to process this sentence because tokenization can be useful" \
| subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg.model

In [ ]:
!subword-nmt learn-bpe -s 1500 < \
{CORPORA_PATH}/gutenberg_plain.txt > \
 {MODELS_PATH}/gutenberg_high.model

In [ ]:
!echo "I need to process this sentence because tokenization can be useful" \
| subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg_high.model

### Aplicandolo a otros corpus: La biblia 📖🇻🇦

In [ ]:
BIBLE_FILE_NAMES = {
    "spa": "spa-x-bible-reinavaleracontemporanea",
    "eng": "eng-x-bible-kingjames",
}

In [ ]:
import requests


def get_bible_corpus(lang: str) -> str:
    """Download bible file corpus from GitHub repo"""
    file_name = BIBLE_FILE_NAMES[lang]
    r = requests.get(
        f"https://raw.githubusercontent.com/ximenina/theturningpoint/main/Detailed/corpora/corpusPBC/{file_name}.txt.clean.txt"
    )
    return r.text


def write_plain_text_corpus(raw_text: str, file_name: str) -> None:
    """Write file text on disk"""
    with open(f"{file_name}.txt", "w") as f:
        f.write(raw_text)

#### Biblia en Inglés

In [ ]:
eng_bible_plain_text = get_bible_corpus("eng")
eng_bible_words = eng_bible_plain_text.lower().replace("\n", " ").split()

In [ ]:
print(eng_bible_words[:10])

In [ ]:
len(eng_bible_words)

In [ ]:
eng_bible_types = Counter(eng_bible_words)

In [ ]:
rprint(eng_bible_types.most_common(30))

In [ ]:
eng_bible_lemmas_types = Counter(lemmatize(eng_bible_words, lang="en"))

In [ ]:
write_plain_text_corpus(eng_bible_plain_text, f"{CORPORA_PATH}/eng-bible")

In [ ]:
!subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg_high.model < \
 {CORPORA_PATH}/eng-bible.txt > \
 {CORPORA_PATH}/eng-bible-tokenized.txt

In [ ]:
with open(f"{CORPORA_PATH}/eng-bible-tokenized.txt", "r") as f:
    tokenized_data = f.read()
eng_bible_tokenized = tokenized_data.split()

In [ ]:
rprint(eng_bible_tokenized[:10])

In [ ]:
len(eng_bible_tokenized)

In [ ]:
eng_bible_tokenized_types = Counter(eng_bible_tokenized)
len(eng_bible_tokenized_types)

In [ ]:
eng_bible_tokenized_types.most_common(30)

#### ¿Qué pasa si aplicamos el modelo aprendido con Gutenberg a otras lenguas?

In [ ]:
spa_bible_plain_text = get_bible_corpus("spa")
spa_bible_words = spa_bible_plain_text.replace("\n", " ").lower().split()

In [ ]:
spa_bible_words[:10]

In [ ]:
len(spa_bible_words)

In [ ]:
spa_bible_types = Counter(spa_bible_words)
len(spa_bible_types)

In [ ]:
spa_bible_types.most_common(30)

In [ ]:
spa_bible_lemmas_types = Counter(lemmatize(spa_bible_words, lang="es"))
len(spa_bible_lemmas_types)

In [ ]:
write_plain_text_corpus(spa_bible_plain_text, f"{CORPORA_PATH}/spa-bible")

In [ ]:
!subword-nmt apply-bpe -c {MODELS_PATH}/gutenberg_high.model < \
 {CORPORA_PATH}/spa-bible.txt > \
 {CORPORA_PATH}/spa-bible-tokenized.txt

In [ ]:
with open(f"{CORPORA_PATH}/spa-bible-tokenized.txt", "r") as f:
    tokenized_text = f.read()
spa_bible_tokenized = tokenized_text.split()

In [ ]:
spa_bible_tokenized[:10]

In [ ]:
len(spa_bible_tokenized)

In [ ]:
spa_bible_tokenized_types = Counter(spa_bible_tokenized)
len(spa_bible_tokenized_types)

In [ ]:
spa_bible_tokenized_types.most_common(40)

## Type-token Ratio (TTR)

- Una forma de medir la variación del vocabulario en un corpus
- Este se calcula como $TTR = \frac{len(types)}{len(tokens)}$
- Puede ser útil para monitorear la variación lexica de un texto

In [ ]:
rprint("Información de la biblia en Inglés")
rprint("Tokens:", len(eng_bible_words))
rprint("Types ([bright_magenta]word-base):", len(eng_bible_types))
rprint("Types ([bright_yellow]lemmatized)", len(eng_bible_lemmas_types))
rprint("Types ([bright_green]BPE):", len(eng_bible_tokenized_types))
rprint("TTR ([bright_magenta]word-base):", len(eng_bible_types) / len(eng_bible_words))
rprint("TTR ([bright_green]BPE):", len(eng_bible_tokenized_types) / len(eng_bible_tokenized))

In [ ]:
rprint("Bible Spanish Information")
rprint("Tokens:", len(spa_bible_words))
rprint("Types ([bright_magenta]word-base):", len(spa_bible_types))
rprint("Types ([bright_yellow]lemmatized)", len(spa_bible_lemmas_types))
rprint("Types ([bright_green]BPE):", len(spa_bible_tokenized_types))
rprint("TTR ([bright_magenta]word-base):", len(spa_bible_types) / len(spa_bible_words))
rprint("TTR ([bright_green]BPE):", len(spa_bible_tokenized_types) / len(spa_bible_tokenized))